In [1]:
import pymysql
import re
import json
import pandas as pd
from sqlalchemy import create_engine
import mysql.connector
from pathlib import Path


Datenbank erstellen

In [2]:
# Pfad zur CSV-Datei
#hier relativen Pfad statt absoluten benutzen, dann muss lediglich die Datenbank im gleichen Ordner liegen
csv_file = csv_file = Path.cwd() /'tbl_eintraege.tab'

rows = []

with open(csv_file, encoding='utf-8') as file:
    for line in file:
        rows.append(line.strip().split('\t')[0:6])  #zur Sicherheit werden hier überstehende Zeilen abgeschnitten

# DataFrame erstellen
df = pd.DataFrame(rows[1:], columns=rows[0])

In [3]:
# Daten in die SQL-Datenbank übertragen
user = 'root'
password = 'test'
host = 'localhost'  
port = '3306'
database = 'test'
engine = create_engine(f'mysql+pymysql://{user}:{password}@{host}:{port}/{database}')


table_name = 'all_data'


df.to_sql(table_name, engine, if_exists='replace', index=False)

RuntimeError: 'cryptography' package is required for sha256_password or caching_sha2_password auth methods

In [7]:
# Testtabelle erstellen 

conn = mysql.connector.connect(
    host='localhost',  
    user='root',  
    password='test',  
    database='test',  
    charset='utf8mb4',  
    collation='utf8mb4_general_ci',
    auth_plugin='mysql_native_password' 
)

cursor = conn.cursor()


create_table_query = """
CREATE TABLE IF NOT EXISTS test_data (
    id INT AUTO_INCREMENT PRIMARY KEY,
    lemma VARCHAR(255),
    grundform VARCHAR(255),
    grammatik VARCHAR(255)
) COLLATE='utf8mb4_general_ci';
"""
cursor.execute(create_table_query)


insert_data_query = """
INSERT INTO test_data (lemma, grundform, grammatik)
SELECT lemma, grundform, grammatik
FROM all_data
GROUP BY grundform, lemma, grammatik;
"""
cursor.execute(insert_data_query)


conn.commit()

cursor.close()
conn.close()


In [8]:
#Verbinden mit Datenbank
connection = pymysql.connect(
    host='localhost',
    user='root',
    password='test',
    database='test'
)

In [9]:
#Testdaten mit einem Buchstaben
def fetch_data(letter):
    with connection.cursor(pymysql.cursors.DictCursor) as cursor:
        cursor.execute(f"SELECT * FROM test_data WHERE grundform LIKE '{letter}%'")
        return cursor.fetchall()


data = fetch_data('F')

In [10]:
#Alle Daten
def fetch_data_all():
    with connection.cursor(pymysql.cursors.DictCursor) as cursor:
        cursor.execute("SELECT * FROM all_data")
        return cursor.fetchall()

data_all = fetch_data_all()

In [11]:
connection.close()

Ableitungen:

In [12]:
# Listen der Präfixe, Zirkumfixe und Suffixe
prefixes = [
    "a", "an", "ar", "ab", "aber", "after", "anti", "auf", "aus", "auto", "be", "bei", "bio", "de", "des", "dis",
    "durch", "ein", "emp", "ent", "entgegen", "er", "erz", "ex", "fehl", "fest", "fort", "ge", "gegen", "geo",
    "graf", "her", "herunter", "hin", "hinter", "hyper", "ident", "in", "im", "il", "ir", "inne", "inter", "ko", "kol",
    "kom", "kon", "kor", "kund", "los", "makro", "maxi", "mega", "mikro", "mini", "miss", "mit", "mono", "multi",
    "nach", "nano", "naut", "neo", "non", "para", "pflichtig", "phil", "phob", "poly", "post", "prä", "pro", "proto",
    "pseudo", "quasi", "re", "riesen", "rück", "schwieger", "semi", "stereo", "stief", "tele", "therm", "thermo",
    "trans", "ultra", "um", "un", "unter", "ur", "ver", "vize", "vor", "weg", "wett", "wider", "zer", "zu", "zurecht",
    "zurück", "zusammen", "zuwider", "zwischen"
]

circumfixes = [
    "be...ig", "be...t", "ge...e", "ge...ig", "ge...sel", "ge...t", "ver...ig"
]

suffixes = [
    "a", "abel", "ibel", "ade", "iade", "age", "aholic", "oholic", "oholiker", "aille", "al", "ell", "ament",
    "ement", "an", "and", "ant", "ent", "ante", "ente", "anz", "enz", "ar", "är", "arium", "arm", "artig", "ast", "at",
    "ee", "ei", "eierei", "el", "elchen", "erchen", "elle", "ens", "er", "erich", "erie", "ern", "esk", "ess", "esse",
    "isse", "ette", "eur", "euse", "fach", "fähig", "bold", "chen", "dings", "drom", "e", "i", "ian", "jan", "ice",
    "icht", "ie", "ier", "ieren", "ifizier", "isier", "iere", "ig", "ik", "iker", "ine", "ing", "ingen", "en", "ern",
    "er", "e", "ell", "ion", "tion", "ation", "ismus", "asmus", "ist", "it", "ität", "itis", "iv", "ativ", "ke", "lei",
    "lein", "lekt", "ler", "lich", "ling", "lings", "lon", "los", "mals", "maßen", "mäßig", "mini", "n", "nis", "o",
    "oid", "ol", "or", "ator", "itor", "os", "ös", "ose", "ow", "pflichtig", "reich", "rich", "sal", "sam", "schaft",
    "sche", "seitig", "sel", "sen", "skop", "tex", "thek", "trächtig", "tum", "ung", "ur", "voll", "wang", "wangen",
    "wart", "wärts", "weg", "weise", "werk", "wesen", "zid"
]

umlautungen_und_ablautungen = {
    'a': ['ä', 'i', 'e', 'u', 'ie'],
    'e': ['a', 'o', 'ie', 'u'],
    'i': ['a', 'o', 'ie', 'u'],
    'o': ['e', 'ö', 'a', 'ie'],
    'u': ['o', 'ü', 'ie'],
    'au': ['äu', 'o', 'ie'],
    'ei': ['i', 'ie'],
    'ie': ['o'],
    'ö': ['o']
}
doppelkonsonanten = {
    'b': 'bb',
    'd': 'dd',
    'g': 'gg',
    'k': 'kk',
    'l': 'll',
    'm': 'mm',
    'n': 'nn',
    'p': 'pp',
    'r': 'rr',
    's': 'ss',
    't': 'tt'
}

def generate_variants(word, mappings):
    variants = set()
    for original, changes in mappings.items():
        if isinstance(changes, list):
            for change in changes:
                if original in word:
                    variants.add(word.replace(original, change))
        else:
            if original in word:
                variants.add(word.replace(original, changes))
    return variants

def find_ableitungen(data):
    ableitungen_dict = {}
    
    grundformen_lemma = {entry['grundform'].lower(): entry['lemma'].lower() for entry in data if entry['grundform'] is not None and entry['lemma'] is not None}
    grundformen_original = {entry['grundform'].lower(): entry['grundform'] for entry in data if entry['grundform'] is not None}

    def add_ableitungen(grundform, ableitungen_set):
        lower_grundform = grundform.lower()
        grundform_lemma = grundformen_lemma[lower_grundform]

        umlaut_ablaut_variants = generate_variants(lower_grundform, umlautungen_und_ablautungen)
        doppelkonsonanten_variants = generate_variants(lower_grundform, doppelkonsonanten)

        queue = [lower_grundform]
        processed = set()

        while queue:
            current_form = queue.pop(0)
            current_variants = {current_form}.union(umlaut_ablaut_variants).union(doppelkonsonanten_variants)

            for variant in current_variants:
                for word in data:
                    if not word.get('grundform') or not word.get('lemma'):
                        continue
                    word_form = word['grundform']
                    word_lemma = word['lemma'].lower()
                    lower_word_form = word_form.lower()

                    if lower_word_form != variant and grundformen_lemma.get(lower_word_form) == grundform_lemma:
                        is_derivative = False

                        # Prüfen auf Präfixe
                        for prefix in prefixes:
                            if lower_word_form.startswith(prefix) and lower_word_form[len(prefix):] == variant:
                                is_derivative = True
                                break

                        # Prüfen auf Zirkumfixe
                        if not is_derivative:
                            for circumfix in circumfixes:
                                parts = circumfix.split("...")
                                if lower_word_form.startswith(parts[0]) and lower_word_form.endswith(parts[1]) and lower_word_form[len(parts[0]):-len(parts[1])] == variant:
                                    is_derivative = True
                                    break

                        # Prüfen auf Suffixe
                        if not is_derivative:
                            for suffix in suffixes:
                                if lower_word_form.endswith(suffix) and lower_word_form[:-len(suffix)] == variant:
                                    is_derivative = True
                                    break

                        # Überprüfen, ob die Kombination in der Datenbank existiert
                        if is_derivative:
                            if any(entry.get('grundform') and entry.get('lemma') and entry['grundform'].lower() == lower_word_form and entry['lemma'].lower() != grundform_lemma for entry in data):
                                is_derivative = False

                        if is_derivative and lower_word_form not in ableitungen_set and lower_word_form not in processed:
                            ableitungen_set.add(word_form)
                            queue.append(lower_word_form)
                            processed.add(lower_word_form)

    for entry in data:
        if not entry.get('grundform') or not entry.get('lemma'):
            continue
        grundform = entry['grundform']
        if grundform.lower() == entry['lemma'].lower():
            if grundform.lower() not in ableitungen_dict:
                ableitungen_dict[grundform.lower()] = set()
            add_ableitungen(grundform, ableitungen_dict[grundform.lower()])

    result = {grundformen_original[grundform]: list(forms) for grundform, forms in ableitungen_dict.items() if forms}

    return result


In [13]:
# Test mit einem Buchstaben
ableitungen = find_ableitungen(data)
ableitungen

{'Fabrik': ['Fabriker'],
 'Fach': ['Fächlein'],
 'Fack': ['Facke', 'Fackelein'],
 'Fahne': ['Fahnelein'],
 'Fahrt': ['Fährtel', 'Fährte', 'Fährtlein'],
 'Farn': ['Farnling'],
 'Faser': ['Faserlein', 'fasern'],
 'Fass': ['Fässel', 'Fässchen', 'Fässlein'],
 'faul': ['Faule', 'faulen', 'Fauler', 'faulig'],
 'Faust': ['Fäustlein', 'Fäuster', 'Fäustling', 'Fäustel'],
 'Fechser': ['Fechserlein'],
 'Feder': ['federn', 'Federig'],
 'Feile': ['feilen', 'Feilelein'],
 'Feim': ['Feimlein', 'Feimen'],
 'fein': ['Feiner', 'Feine', 'feinig'],
 'Feind': ['Feindschaft', 'feindlich'],
 'Feld': ['Feldel'],
 'Felge': ['Felgelein'],
 'Fels': ['Felsen', 'felsig'],
 'Femel': ['femelig'],
 'Fenster': ['fenstern', 'Fensterchen'],
 'Ferkel': ['Ferkelchen', 'ferkeln'],
 'Ferse': ['fersen'],
 'Fessel': ['Fesseln'],
 'fest': ['Festung', 'Feste', 'Fester'],
 'Fett': ['fettig', 'Fetter', 'Fetten', 'Fette'],
 'feucht': ['Feuchte'],
 'Feuer': ['Feuerlein', 'feuern'],
 'Fichte': ['fichten'],
 'Fieber': ['Fieberer', 'f

In [14]:
# Speichern in einer JSON-Datei
with open('ableitungen.json', 'w', encoding='utf-8') as json_file:
    json.dump(ableitungen, json_file, ensure_ascii=False, indent=4)


Komposita:

In [15]:
def find_komposita(data):
    grundformen_komposita = {}

    # Schleife zum Sammeln der Komposita
    for entry in data:
        if not entry['grundform']:
            continue
        grundform = entry['grundform'].lower()


        if grundform not in grundformen_komposita:
            grundformen_komposita[grundform] = set()

        for word_entry in data:
            if not word_entry['grundform']:
                continue
            if entry == word_entry:
                continue
            if ' ' in word_entry['grundform']:
                continue
            if not entry['grammatik'] or not word_entry['grammatik']: 
                continue
            if entry['grammatik'][0].lower() != word_entry['grammatik'][0].lower():
                continue
            if entry['lemma'] != word_entry['lemma']:
                continue
        
            word_form = word_entry['grundform'].lower()

            # Überprüfen, ob word_form mit der grundform endet und nicht identisch mit der grundform ist
            if word_form.endswith(grundform) and word_form != grundform:
                # Überprüfen, ob der Rest des Wortes ein Präfix ist
                remaining_part = word_form[:-len(grundform)]
                if remaining_part in prefixes:
                    # Überprüfen, ob die Grundform mit einem passenden Lemma in der Datenbank existiert
                    for db_entry in data:
                        if db_entry['grundform'].lower() == grundform and db_entry['lemma'].lower() == entry['lemma'].lower():
                            grundformen_komposita[grundform].add(word_entry['grundform'])
                            break
                else:
                        # Kein Präfix, das Wort ist ebenfalls ein Kompositum
                    grundformen_komposita[grundform].add(word_entry['grundform'])

    
    result = {}

    grundformen_original = {entry['grundform'].lower(): entry['grundform'] for entry in data}


    result = {grundformen_original[grundform]: list(forms) for grundform, forms in grundformen_komposita.items() if forms}

    return result

In [16]:
# Test mit einem Buchstaben
komposita = find_komposita(data)
komposita

{'Frostblume': ['Fensterfrostblume'],
 'fahren': ['fürfahren', 'fürhinfahren', 'Fahrradfahren', 'fortfahren'],
 'Fuhre': ['Feldfuhre'],
 'Falle': ['Flöhfalle'],
 'Falte': ['Fressfalte', 'Fettfalte'],
 'farbig': ['fleischfarbig'],
 'Fass': ['Fleischfass'],
 'Fässlein': ['Futterfässlein', 'Fleischfässlein'],
 'Faxerei': ['Fixfaxerei'],
 'Feder': ['Flaumfeder'],
 'Federhalter': ['Füllfederhalter'],
 'Federlein': ['Frau-Holle-Federlein'],
 'Feile': ['Flachfeile'],
 'Fenster': ['Fürfenster'],
 'fest': ['Feuerwehrfest'],
 'Feste': ['Fronfeste'],
 'Fett': ['Fleischfett'],
 'fetzen': ['fürfetzen'],
 'Feuer': ['Fegefeuer', 'Flugfeuer', 'Fegfeuer', 'Flammfeuer'],
 'Fleck': ['Farbfleck', 'Feuchtfleck', 'Flurfleck', 'Fallfleck'],
 'Fliege': ['Fleischfliege'],
 'fliegen': ['fortfliegen', 'fürfliegen'],
 'Flurer': ['Feldflurer'],
 'Fresser': ['Fettfresser', 'Fleischfresser'],
 'frieren': ['festfrieren'],
 'Fuchser': ['Federnfuchser'],
 'Füller': ['Flaschenfüller'],
 'Furche': ['Fehlfurche'],
 'Furie

In [17]:
# Speichern in einer JSON-Datei
with open('komposita.json', 'w', encoding='utf-8') as json_file:
    json.dump(komposita, json_file, ensure_ascii=False, indent=4)

Hapax Legomena:

In [25]:
def find_hapax_legomena(data):
    frequency = {}
    lemma_map = {}

    for entry in data:
        word = entry['grundform']
        lemma = entry['lemma']

        if word not in frequency:
            frequency[word] = 0
            lemma_map[word] = set()

        frequency[word] += 1
        lemma_map[word].add(lemma)

    hapax_legomena = [
        word for word, count in frequency.items()
        if count == 1 or len(lemma_map[word]) == frequency[word]
    ]

    return hapax_legomena



hapax_legomena = find_hapax_legomena(data_all)


KeyError: 'grundform'

In [24]:
with open('hapax_legomena.json', 'w', encoding='utf-8') as json_file:
    json.dump(hapax_legomena, json_file, ensure_ascii=False, indent=4)


NameError: name 'hapax_legomena' is not defined

Diminutive:

In [26]:
def find_diminutive(data):
    def umlaut_variants(word):
        umlaut_map = {'a': 'ä', 'o': 'ö', 'u': 'ü'}
        variants = {word}
        
        for i, char in enumerate(word):
            if char in umlaut_map:
                umlaut_variant = word[:i] + umlaut_map[char] + word[i + 1:]
                variants.add(umlaut_variant)
        
        for original, umlaut in umlaut_map.items():
            double_vowel = original * 2
            if double_vowel in word:
                variants.add(word.replace(double_vowel, umlaut))

        return variants

    def generate_possible_bases(word):
        variants = umlaut_variants(word)
        variants.add(word)
        if word.endswith(('e', 'l')):
            variants.update(umlaut_variants(word[:-1]))
        if word.endswith(('en', 'el')):
            variants.update(umlaut_variants(word[:-2]))
        return variants

    diminutive_suffixes = ['elchen', 'chen', 'lein', 'la', 'le', 'erl', 'al', 'el', 'rl', 'ele', 'elein', 'ale', 'l', 'i']

    grundformen_diminutive = {}
    diminutive_to_grundform = {}

    grundform_dict = {entry['grundform'].lower(): entry for entry in data if entry['grundform'] is not None}
    
    for entry in data:
        if not entry.get('grundform') or not entry.get('lemma') or not entry.get('grammatik'):
            continue
        
        grundform = entry['grundform']
        grundform_lower = grundform.lower()
        umlaut_variants_set = generate_possible_bases(grundform_lower)
        
        if grundform not in grundformen_diminutive:
            grundformen_diminutive[grundform] = set()

        # Überprüfen, ob es sich um einen Familiennamen handelt
        if entry['grammatik'][0] == 'NaAF':
            continue
        
        for suffix in diminutive_suffixes:
            for variant in umlaut_variants_set:
                diminutive_form = variant + suffix
                diminutive_form_original = [e['grundform'] for e in data if e.get('grundform') and e['grundform'].lower() == diminutive_form]
                
                if diminutive_form in grundform_dict:
                    diminutive_entry = grundform_dict[diminutive_form]
                    if diminutive_entry.get('lemma') == entry['lemma'] and diminutive_entry.get('grammatik')[0] == entry['grammatik'][0]:
                        if diminutive_form != grundform_lower:
                            grundformen_diminutive[grundform].add(diminutive_form_original[0])
                            diminutive_to_grundform[diminutive_form_original[0]] = grundform

    for diminutive, base in diminutive_to_grundform.items():
        for dim in grundformen_diminutive[diminutive]:
            grundformen_diminutive[base].add(dim)
        grundformen_diminutive[diminutive] = set()

    filtered_grundformen_diminutive = {key: list(value) for key, value in grundformen_diminutive.items() if value}

    return filtered_grundformen_diminutive



In [27]:
# Test mit einem Buchstaben
diminutive = find_diminutive(data)
diminutive

TypeError: 'NoneType' object is not subscriptable

In [ ]:
# Speichern in einer JSON-Datei
with open('diminutive.json', 'w', encoding='utf-8') as json_file:
    json.dump(diminutive, json_file, ensure_ascii=False, indent=4)